# Reading the journal

Load what a run produced, to explore it by hand. Read only: nothing here changes the data.
Format of every field: `docs/journal.md`. Files of a run folder:

- `runs.jsonl`: one line per launch of the bot (versions, world seed, difficulty...)
- `transitions.jsonl`: one line per skill run (state before, skill, trace, state after, result); the steps
  inside a shortcut have their own line, with `parent_seq`
- `episodes.jsonl`: the LLM episodes (only in the folder of a bot started with `serve`)


In [1]:
import collections
import json
import pathlib
import sys

# The repo root, whether the notebook runs from notebooks/ or from the root (and `brain` importable)
ROOT = (
    pathlib.Path.cwd().parent
    if pathlib.Path.cwd().name == "notebooks"
    else pathlib.Path.cwd()
)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

# The collection runs, newest last
for folder in sorted((DATA / "collection").glob("run-*")):
    sizes = {f.name: f"{f.stat().st_size / 1e6:.0f} MB" for f in folder.glob("*.jsonl")}
    print(folder.name, sizes)

run-20261113-030804-0746 {'runs.jsonl': '0 MB', 'transitions.jsonl': '163 MB'}
run-20261113-035536-0ad5 {'runs.jsonl': '0 MB', 'transitions.jsonl': '37 MB', 'episodes.jsonl': '2 MB'}


## Choose what to read

A collection run has its own folder. In `data/dev/` every launch shares the same files: give `RUN_ID`
to keep only one launch (`seq` starts again at 0 at each launch).


In [2]:
RUN_DIR = sorted((DATA / "collection").glob("run-*"))[
    -1
]  # the newest collection run; or DATA / "dev"
RUN_ID = None  # only for DATA / "dev": e.g. "run-20261113-023339-6839"
LIGHT = True  # leave out the heaviest parts (block grids, traces) to save memory; False = everything
print(RUN_DIR)

/Users/tombourdareau/Projects/mc-agent/data/collection/run-20261113-035536-0ad5


In [3]:
def read_jsonl(path, run_id=None, light=False):
    """The lines of a JSONL file as dicts (only one run if run_id is given)."""
    if not path.exists():
        return []
    lines = []
    with path.open() as f:
        for text in f:
            if run_id and run_id not in text[:300]:
                continue  # quick text check before parsing (lines are big)
            line = json.loads(text)
            if run_id and line.get("run_id") != run_id:
                continue
            if light:
                for state in ("state", "next_state"):
                    if line.get(state):
                        line[state].pop("blocks", None)
                line.pop("trace", None)
            lines.append(line)
    return lines


runs = read_jsonl(RUN_DIR / "runs.jsonl", RUN_ID)
transitions = read_jsonl(RUN_DIR / "transitions.jsonl", RUN_ID, light=LIGHT)
episodes = read_jsonl(RUN_DIR / "episodes.jsonl")
print(
    len(runs),
    "runs,",
    len(transitions),
    "transitions,",
    len(episodes),
    "episode events",
)

1 runs, 204 transitions, 90 episode events


In [4]:
# The launch: versions, world seed, difficulty
runs[-1]

{'schema_version': 1,
 'type': 'run_start',
 'run_id': 'run-20261113-035536-0ad5',
 'ts': '2026-11-13T03:55:36.894Z',
 'purpose': 'collection',
 'data_dir': '/Users/tombourdareau/Projects/mc-agent/data/collection/run-20261113-035536-0ad5',
 'bot': 'Agent',
 'world_id': 'localhost:25565',
 'world_seed': '4061568074923464531',
 'versions': {'code': '940419b',
  'minecraft': '1.21.11',
  'server': 'This server is running Paper version 1.21.11-132-ver/1.21.11@c5eb079 (2026-05-11T11:43:09Z) (Implementing API version 1.21.11-R0.1-SNAPSHOT)\nYou are running the latest version',
  'mineflayer': '4.39.0',
  'pathfinder': '2.4.5',
  'node': 'v26.5.0'},
 'game': {'difficulty': 'easy',
  'game_mode': 'survival',
  'dimension': 'overworld',
  'tick_rate': 20,
  'view_distance': 3,
  'simulation_distance': 3},
 'host': 'MacBook-Air-de-Tom-2.local'}

## Quick look


In [5]:
# How each skill ended (top lines only: the skills that were asked for, not their steps)
top = [t for t in transitions if t.get("parent_seq") is None]
counts = collections.Counter((t["skill"]["name"], t["reason"]) for t in top)
for (skill, reason), n in sorted(counts.items()):
    print(f"{skill:15} {reason:22} {n}")

craft           missing_ingredients    11
craft           ok                     18
explore         ok                     9
explore         timeout                1
find            not_found              9
find            ok                     1
mine            block_not_found        2
mine            ok                     14
recipe          invalid_args           1
recipe          ok                     10


In [6]:
def short(t):
    """One line, in short: what was asked and how it ended."""
    return {
        k: v
        for k, v in {
            "seq": t["seq"],
            "parent": t.get("parent_seq"),
            "skill": t["skill"]["name"],
            "args": t["skill"]["args"],
            "success": t["success"],
            "reason": t["reason"],
            "s": round(t["duration"], 1),
        }.items()
        if v is not None
    }


def line(seq):
    """The whole line with this seq."""
    return next(t for t in transitions if t["seq"] == seq)


def steps(seq):
    """The steps run inside the skill of line `seq` (e.g. mine -> find, collect_block...)."""
    return [short(t) for t in transitions if t.get("parent_seq") == seq]


def change(seq):
    """What the inventory gained or lost during line `seq`."""
    t = line(seq)
    before, after = (
        (t["state"] or {}).get("inventory", {}),
        (t["next_state"] or {}).get("inventory", {}),
    )
    return {
        k: after.get(k, 0) - before.get(k, 0)
        for k in before | after
        if after.get(k, 0) != before.get(k, 0)
    }


for t in top[:15]:
    print(short(t))

{'seq': 0, 'skill': 'recipe', 'args': {'item': 'crafting_table'}, 'success': True, 'reason': 'ok', 's': 0.1}
{'seq': 1, 'skill': 'craft', 'args': {'item': 'crafting_table'}, 'success': False, 'reason': 'missing_ingredients', 's': 0.0}
{'seq': 2, 'skill': 'recipe', 'args': {'item': '#planks'}, 'success': True, 'reason': 'ok', 's': 0.0}
{'seq': 3, 'skill': 'mine', 'args': {'count': 1, 'block': 'oak_log'}, 'success': True, 'reason': 'ok', 's': 23.5}
{'seq': 8, 'skill': 'craft', 'args': {'item': 'oak_planks', 'count': 4}, 'success': True, 'reason': 'ok', 's': 0.4}
{'seq': 9, 'skill': 'craft', 'args': {'item': 'crafting_table'}, 'success': True, 'reason': 'ok', 's': 0.4}
{'seq': 10, 'skill': 'craft', 'args': {'item': 'crafting_table'}, 'success': False, 'reason': 'missing_ingredients', 's': 0.0}
{'seq': 11, 'skill': 'mine', 'args': {'block': 'oak_log', 'count': 1}, 'success': False, 'reason': 'block_not_found', 's': 0.1}
{'seq': 13, 'skill': 'explore', 'args': {'direction': 'north'}, 'succe

In [7]:
# The failures, with their details
for t in top:
    if not t["success"]:
        print(short(t), t["details"])

{'seq': 1, 'skill': 'craft', 'args': {'item': 'crafting_table'}, 'success': False, 'reason': 'missing_ingredients', 's': 0.0} {'item': 'crafting_table', 'count': 1, 'needs_table': False, 'makes': 1, 'ingredients': [{'group': '#planks', 'one_of': ['cherry_planks', 'bamboo_planks', 'mangrove_planks', 'warped_planks', 'crimson_planks', 'pale_oak_planks', 'dark_oak_planks', 'acacia_planks', 'jungle_planks', 'birch_planks', 'spruce_planks', 'oak_planks'], 'required': 4, 'owned': 0}]}
{'seq': 10, 'skill': 'craft', 'args': {'item': 'crafting_table'}, 'success': False, 'reason': 'missing_ingredients', 's': 0.0} {'item': 'crafting_table', 'count': 1, 'needs_table': False, 'makes': 1, 'ingredients': [{'group': '#planks', 'one_of': ['cherry_planks', 'bamboo_planks', 'mangrove_planks', 'warped_planks', 'crimson_planks', 'pale_oak_planks', 'dark_oak_planks', 'acacia_planks', 'jungle_planks', 'birch_planks', 'spruce_planks', 'oak_planks'], 'required': 4, 'owned': 0}]}
{'seq': 11, 'skill': 'mine', 'a

In [8]:
# One line in detail: change the seq
seq = top[0]["seq"]
print(short(line(seq)))
print("inventory change:", change(seq))
print("details:", line(seq)["details"])
print("steps:", steps(seq))

{'seq': 0, 'skill': 'recipe', 'args': {'item': 'crafting_table'}, 'success': True, 'reason': 'ok', 's': 0.1}
inventory change: {}
details: {'item': 'crafting_table', 'needs_table': False, 'makes': 1, 'ingredients': [{'group': '#planks', 'one_of': ['cherry_planks', 'bamboo_planks', 'mangrove_planks', 'warped_planks', 'crimson_planks', 'pale_oak_planks', 'dark_oak_planks', 'acacia_planks', 'jungle_planks', 'birch_planks', 'spruce_planks', 'oak_planks'], 'required': 4, 'owned': 0}]}
steps: []


## Episodes (LLM)

Only in the folder of a bot started with `serve`. The same view as `uv run python -m brain.show`.


In [9]:
from brain.show import load_episodes, load_transitions, show

if (RUN_DIR / "episodes.jsonl").exists():
    by_episode = load_episodes(RUN_DIR / "episodes.jsonl")
    for episode_id, events in by_episode.items():
        end = events[-1].get("outcome", {})
        print(episode_id, events[0].get("goal"), "->", end.get("reason"))
else:
    print("no episodes in", RUN_DIR)

ep-20261113-035600-ccc0 craft a crafting table -> condition_met
ep-20261113-035843-7726 craft a crafting table -> condition_met
ep-20261113-040410-a35f craft a crafting table -> condition_met
ep-20261113-040622-e66e craft a crafting table -> condition_met
ep-20261113-040834-7efd craft a crafting table -> condition_met
ep-20261113-040957-be69 craft a crafting table -> condition_met
ep-20261113-041258-f152 craft a crafting table -> max_turns
ep-20261113-042309-c1c6 craft a crafting table -> condition_met
ep-20261113-042536-5179 craft a crafting table -> condition_met
ep-20261113-042810-04b0 craft a crafting table -> condition_met
ep-20261113-053751-3439 collect 3 oak logs -> condition_met
ep-20261113-053915-38ae collect 3 oak logs -> condition_met
ep-20261113-053952-b6b4 collect 3 oak logs -> condition_met
ep-20261113-054052-dddf collect 3 oak logs -> condition_met
ep-20261113-054131-69e9 collect 3 oak logs -> condition_met


In [10]:
# One episode, turn by turn: change the index (-1 = the last one)
if (RUN_DIR / "episodes.jsonl").exists():
    episode_id = list(by_episode)[-1]
    print(
        show(
            by_episode[episode_id],
            load_transitions(RUN_DIR / "transitions.jsonl", episode_id),
        )
    )

EPISODE ep-20261113-054131-69e9  goal: 'collect 3 oak logs'
  model qwen2.5:14b, context 16384, code 940419b, max 10 turns

turn 0  [343 tokens, 4.922 s]
  goal condition: {'oak_log': 3}

turn 1  [2604 tokens, 13.833 s]
  -> mine(block='oak_log', count=3)  ok
       {"dug": 3, "collected": {"oak_log": 3}, "unreachable": 0}

END  SUCCESS (condition_met)  2 turns, 1 skill calls, 43.638 s
  final inventory: {'oak_log': 3}
